This notebook repeats the Optuna search multiple times and saves the results. This procedure may take very long, so users are advised to run this notebook in a terminal session, for e.g. tmux.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "network_generator").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from network_generator.config import NetworkConfig, TargetConfig, TrainingConfig

network_config = NetworkConfig(num_firms=100_000)
target_config = TargetConfig()
training_config = TrainingConfig()

In [ ]:
from network_generator.optuna_search.run_multiple_times import (
    run_optuna_theta_multiple_experiments,
)

df_thetas = run_optuna_theta_multiple_experiments(
    num_experiments=training_config.num_experiments,
    network_config=network_config,
    target_config=target_config,
    data_path="./data/",
    n_trials=training_config.n_trials,
    theta_k_in_upper_bound=0.5,
    theta_k_in_lower_bound=-0.5,
    theta_k_out_upper_bound=-0.5,
    theta_k_out_lower_bound=-1.0,
    theta_f_in_upper_bound=-1.0,
    theta_f_in_lower_bound=-1.5,
    theta_f_out_upper_bound=-1.0,
    theta_f_out_lower_bound=-1.5,
    # theta_k_in_upper_bound = training_config.theta_k_in_upper_bound,
    # theta_k_in_lower_bound = training_config.theta_k_in_lower_bound,
    # theta_k_out_upper_bound = training_config.theta_k_out_upper_bound,
    # theta_k_out_lower_bound = training_config.theta_k_out_lower_bound,
    # theta_f_in_upper_bound = training_config.theta_f_in_upper_bound,
    # theta_f_in_lower_bound = training_config.theta_f_in_lower_bound,
    # theta_f_out_upper_bound = training_config.theta_f_out_upper_bound,
    # theta_f_out_lower_bound = training_config.theta_f_out_lower_bound,
)

In [ ]:
# save the results from repeating the Optuna search multiple times
df_thetas.to_csv('./paper/artifacts/tuning_csvs/best_thetas_200expts_restricted_bounds_zeta-0.14.csv')

In [ ]:
# load the saved results

import pandas as pd

# this version of the dataframe was from searching over the space [-2, 2] x [-2, 2] x [-2, 2] x [-2, 2]. We reported this version in the S.I.
df_thetas = pd.read_csv('./paper/artifacts/tuning_csvs/best_thetas_200expts_full_bounds_zeta-0.14.csv', index_col = 'Unnamed: 0')

# this version of the dataframe was from searching over the space [-0.5, 0.5] x [-1, -0.5] x [-1.5, -1] x [-1.5, -1], i.e. smaller.
# this version is just for further exploratory analysis, the figure does not appear in the paper.
# df_thetas = pd.read_csv('./paper/artifacts/tuning_csvs/best_thetas_200expts_restricted_bounds_zeta-0.14.csv', index_col = 'Unnamed: 0')

df_thetas

In [ ]:
# Visualize the distribution of the estimated theta parameters

import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(3, 2)

# First row
ax00 = fig.add_subplot(gs[0, 0])
ax01 = fig.add_subplot(gs[0, 1])

# Second row
ax10 = fig.add_subplot(gs[1, 0])
ax11 = fig.add_subplot(gs[1, 1])

# Third row (span both columns)
ax2 = fig.add_subplot(gs[2, :])

# Histograms
ax00.hist(df_thetas.loc['theta_k_in'][:-2], bins=100, color='blue')
ax00.set_ylabel('Count',)
ax00.set_xlabel(r'$\theta_{k,\mathrm{in}}$', )

ax01.hist(df_thetas.loc['theta_k_out'][:-2], bins=100, color='blue')
ax01.set_ylabel('Count',)
ax01.set_xlabel(r'$\theta_{k,\mathrm{out}}$', )

ax10.hist(df_thetas.loc['theta_f_in'][:-2], bins=100, color='blue')
ax10.set_ylabel('Count',)
ax10.set_xlabel(r'$\theta_{f,\mathrm{in}}$',)

ax11.hist(df_thetas.loc['theta_f_out'][:-2], bins=100, color='blue')
ax11.set_ylabel('Count',)
ax11.set_xlabel(r'$\theta_{f,\mathrm{out}}$',)

ax2.scatter(
    df_thetas.loc['theta_f_in'][:-2],
    df_thetas.loc['theta_f_out'][:-2],
    marker = '.',
    color='blue'
)

ax2.set_xlabel(r'$\theta_{f,\mathrm{in}}$',)
ax2.set_ylabel(r'$\theta_{f,\mathrm{out}}$',)
ax2.set_title(r'Scatterplot of $\theta_{f,\mathrm{in}}$ against $\theta_{f,\mathrm{out}}$',)

plt.tight_layout()
plt.show()

In [ ]:
len(df_thetas.loc['theta_f_in'][:-2][ df_thetas.loc['theta_f_in'][:-2] < 0]), len(df_thetas.loc['theta_f_in'][:-2][ df_thetas.loc['theta_f_in'][:-2] > 0])

In [ ]:
df_thetas.loc['theta_f_in'][:-2][ df_thetas.loc['theta_f_in'][:-2] < 0].mean(),df_thetas.loc['theta_f_in'][:-2][ df_thetas.loc['theta_f_in'][:-2] < 0].median()

In [ ]:
df_thetas.loc['theta_f_in'][:-2][ df_thetas.loc['theta_f_in'][:-2] > 0].mean(),df_thetas.loc['theta_f_in'][:-2][ df_thetas.loc['theta_f_in'][:-2] > 0].median(),

In [ ]:
df_thetas.loc['theta_f_out'][:-2][df_thetas.loc['theta_f_out'][:-2] < 0].mean(), df_thetas.loc['theta_f_out'][:-2][df_thetas.loc['theta_f_out'][:-2] < 0].median() 

In [ ]:
df_thetas.loc['theta_f_out'][:-2][df_thetas.loc['theta_f_out'][:-2] >0].mean(), df_thetas.loc['theta_f_out'][:-2][df_thetas.loc['theta_f_out'][:-2] >0].median()

In [ ]:
# print the latex table of the best theta parameters

from paper.scripts.best_theta_parameters_latex_table import (
    dataframe_to_best_theta_parameters_latextable,
)

print(dataframe_to_best_theta_parameters_latextable(df_thetas))